# NB09 – Revenue Stacking: VPP, FCR/aFRR & Smart Tariff
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Reine Grid-Arbitrage reicht nicht. Batterien können mehrere Erlösquellen  
**gleichzeitig erschliessen** — dieser Notebook quantifiziert den Mehrwert.

**Erlösquellen:**
- FCR (Primärreserve) — automatische Frequenzstabilisierung
- aFRR (Sekundärreserve) — koordinierter Dispatch auf Swissgrid-Signal
- Kapazitätsprämie (VPP) — aggregierte Leistungsbereitstellung
- Smart Tariff — reduziertes Netzentgelt bei Lastkoordination

---
| [← NB08 Marktdynamik](08_Marktdynamik.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB10 Dispatch-Optimierung](10_Dispatch_Optimierung.ipynb) |
|:---|:---:|---:|


---
## 1. Daten laden

### Voraussetzungen
| Datei | Erzeugt in | Inhalt |
|-------|-----------|--------|
| `wirtschaftlichkeit.csv` | NB02 Sektion 4 | ROI/CAPEX aus reiner Arbitrage |
| `spread_zeitreihe.csv` | NB02 Sektion 7 | Monatlicher Spread (Basiserlös-Indikator) |

**Keine neuen externen Daten nötig** — Erlösstacking-Berechnung basiert auf  
Literaturwerten (Swissgrid FCR/aFRR Ausschreibungsdaten, EPEX SPOT Intraday).


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, warnings, json as _json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

with open('config.json') as _f:
    CFG = _json.load(_f)

MODE         = CFG['mode']
DIR_INTER    = os.path.join(MODE, 'intermediate')
SZ_AKTIV     = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_INTER_SZ = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR      = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: config.json
_viz=CFG.get('visualisierung',{}).get('farben',{})
BG_DARK  = _viz.get('bg_dark',  '#0d1117')
BG_PANEL = _viz.get('bg_panel', '#141414')
C_PRICE  = _viz.get('c_price',  '#FFA726')
C_LOAD   = _viz.get('c_load',   '#66BB6A')
C_CHARGE = _viz.get('c_charge', '#1565C0')
C_FEED   = _viz.get('c_feed',   '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])

LIFETIME = CFG['pflicht']['wirtschaftlichkeit']['lifetime_j']  # = 12 J (konservativer Mittelwert Li-Ion CH)

print(f'MODE     : {MODE}')
print(f'Szenario : {SZ_AKTIV}')
# -- Transfer: Ergebnisse aus transfer.json laden ----------------------------
_tf_path = 'transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    TF      = _json.load(open(_tf_path))
    _dt     = TF.get('datenzeitraum', {})
    _sim    = TF.get('simulation', {})
    TF_N_YEARS  = _dt.get('n_years', None)
    TF_START    = _dt.get('start_date', 'unbekannt')
    TF_END      = _dt.get('end_date', 'unbekannt')
    TF_SPREAD   = _sim.get('spread_mean_eur_mwh', None)
    TF_ECON     = _sim.get('wirtschaftlichkeit', {})
    TF_HYB      = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    TF_KUER     = CFG.get('kuer_aktiv', {})   # aus config.json (SSOT)
    print(f"transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")
else:
    TF = {}; TF_N_YEARS = None; TF_START = TF_END = 'unbekannt'
    TF_SPREAD = None; TF_ECON = {}; TF_HYB = {}; TF_KUER = {}
    print('⚠️  transfer.json nicht gefunden — NB01/NB02 zuerst ausführen')

In [ ]:
# ── Basis-Wirtschaftlichkeit laden ────────────────────────────────────────────
ECON_FILE   = os.path.join(DIR_INTER_SZ, 'wirtschaftlichkeit.csv')
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')

if not os.path.exists(ECON_FILE):
    raise FileNotFoundError(f'{ECON_FILE} fehlt → NB02 Sektion 4 ausführen.')

df_econ = pd.read_csv(ECON_FILE)
df_sp   = pd.read_csv(SPREAD_FILE) if os.path.exists(SPREAD_FILE) else None

print('Basis-Wirtschaftlichkeit (reine Arbitrage):')
print(df_econ[['segment','annual_rev','net_annual','payback_years','roi_pct']].to_string(index=False))


---
## 2. Analyse: Erlösquellen & Stacking-Potenzial

*Aus NB07_Erweiterungen Sektion 4 — hier vertieft und quantifiziert.*

### Erlösquellen-Tabelle (Literaturwerte)

| Quelle | Mechanismus | Zusatzerlös [EUR/kWh/Jahr] | Status CH |
|---|---|---|---|
| **FCR** | Automatische Frequenzstabilisierung | +30–80 | Verfügbar (Swissgrid) |
| **aFRR** | Koordinierter Dispatch | +20–50 | Verfügbar (aggregiert) |
| **Kapazitätsprämie (VPP)** | Aggregierte Leistungsbereitstellung | +20–40 | In Aufbau |
| **Smart Tariff** | Reduziertes Netzentgelt | +15–30% der Arbitrage | Pilotphase |

### Internationale Referenzmodelle

| Land | Anbieter | Modell | Status |
|---|---|---|---|
| UK | Octopus Energy, OVO | Batterie subventioniert + Smart Tarife | Operativ |
| DE | Next Kraftwerke, Sonnen | Aggregierter Pool, FCR/aFRR | Operativ |
| AT | Wien Energie | Flex Pool, Direktzahlung | Pilotphase |
| CH | Swissgrid | Flexibilitätsmarkt für aggregierte Ressourcen | ~2026–2028 |

**Monitoring-Trigger:** Wenn Swissgrid Flexibilitätsmarkt für Heimspeicher < 100 kWh öffnet →  
sofortiger Einstieg in aggregierte Vermarktung empfohlen.


### 2b. FCR: Verfügbarkeitsprämie — kein Energieerlös

Der **grösste Einzelposten im Stacking** (FCR: 30–80 EUR/kWh/Jahr) folgt einer anderen
Logik als Arbitrage: Swissgrid zahlt für das *Vorhalten* von Regelleistung — nicht für
gelieferte oder aufgenommene Energie. Die Batterie muss nur garantieren, dass sie innerhalb
von 30 Sekunden reagieren kann. Im Extremfall fliesst nie Strom, die Prämie fliesst trotzdem.

In der Praxis aktiviert FCR symmetrisch (gleich viel Laden wie Einspeisen) — der
Netto-Energiefluss nähert sich über einen Tag gegen null, was die Batterie schont.

Das erklärt, warum FCR allein das **4-fache der gesamten Arbitrage** (Privat 10 kWh)
einbringen kann — und warum Revenue Stacking den Break-Even von >50 auf ~3–5 Jahre senkt.

> Ausführliche Erklärung mit Zahlenbeispiel und CH-Disclaimer
> → [NB05 Business Strategy, Sektion 7.3](05_Business_Strategy.ipynb)


In [ ]:
# ── Erlösstacking-Modell ──────────────────────────────────────────────────────
# Literaturbasierte Schätzwerte (EUR/kWh Kapazität pro Jahr)
# Pessimistisch = untere Bandbreite, optimistisch = obere Bandbreite

STACKING = {
    'FCR':            {'pesimistisch': 20, 'mitte': 50,  'optimistisch': 80},
    'aFRR':           {'pesimistisch': 10, 'mitte': 30,  'optimistisch': 50},
    'VPP-Praemie':    {'pesimistisch':  0, 'mitte': 25,  'optimistisch': 40},
    'Smart-Tariff':   {'pesimistisch':  5, 'mitte': 12,  'optimistisch': 20},
}

print('Erlösstacking-Szenario (EUR/kWh Kapazität/Jahr):')
print(f'  {"Quelle":<20} {"Pess.":>8} {"Mitte":>8} {"Opt.":>8}')
print('  ' + '-'*48)
total = {'pesimistisch':0, 'mitte':0, 'optimistisch':0}
for src, vals in STACKING.items():
    print(f'  {src:<20} {vals["pesimistisch"]:>8.0f} {vals["mitte"]:>8.0f} {vals["optimistisch"]:>8.0f}')
    for k in total: total[k] += vals[k]
print('  ' + '-'*48)
print(f'  {"TOTAL Zusatz":<20} {total["pesimistisch"]:>8.0f} {total["mitte"]:>8.0f} {total["optimistisch"]:>8.0f}')
print()

# Angewendet auf alle Segmente
print('Wirkung auf Wirtschaftlichkeit (mittleres Stacking-Szenario):')
print(f'  {"Segment":<22} {"Arbitrage/kWh":>14} {"Stacking/kWh":>13} {"Total/kWh":>10} {"BE alt":>8} {"BE neu":>8}')
print('  ' + '-'*80)
for _, row in df_econ.iterrows():
    arb = row['rev_per_kwh']
    stack = total['mitte']
    total_rev = arb + stack
    capex_kwh = row['capex'] / {'Privat_10kWh':10,'Gewerbe_100kWh':100,
                                 'Industrie_1MWh':1000,'Utility_10MWh':10000}[row['segment']]
    opex_kwh  = capex_kwh * 0.015
    net_old = arb - opex_kwh
    net_new = total_rev - opex_kwh
    be_old = capex_kwh / net_old if net_old > 0 else 99
    be_new = capex_kwh / net_new if net_new > 0 else 99
    be_old_s = f'{be_old:.0f}J' if be_old < 50 else '>50J'
    be_new_s = f'{be_new:.0f}J' if be_new < 50 else '>50J'
    print(f'  {row["segment"]:<22} {arb:>13.2f} {stack:>13.0f} {total_rev:>10.2f} {be_old_s:>8} {be_new_s:>8}')


In [ ]:
# ── Chart 09b: FCR als Gamechanger — Break-Even pro Segment ─────────────────
# Zeigt: FCR allein (Verfügbarkeitsprämie) übertrifft jede Arbitrage-Optimierung
import matplotlib.patches as mpatches

SCENARIOS_BAR = [
    ('Nur Arbitrage',              0,    '#EF5350'),
    ('+ FCR (Swissgrid, Mitte)',   50,    '#FFA726'),
    ('+ FCR + aFRR',              80,    '#66BB6A'),
    ('Voll-Stacking (Mitte)',     117,    '#5DCAA5'),
]

segs      = list(df_econ['segment'])
seg_short = [s.split('_')[0] for s in segs]
SEG_KWH   = {'Privat_10kWh':10,'Gewerbe_100kWh':100,
             'Industrie_1MWh':1000,'Utility_10MWh':10000}
capex_kwh_vals = [row['capex'] / SEG_KWH[row['segment']]
                  for _, row in df_econ.iterrows()]
arb_vals = list(df_econ['rev_per_kwh'])

n_scen = len(SCENARIOS_BAR)
x = np.arange(len(segs))
width = 0.18

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG_DARK)
for ax in axes:
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors='#bbbbbb')
    for sp in ax.spines.values(): sp.set_edgecolor('#333333')
fig.suptitle('FCR als Gamechanger: Break-Even und ROI pro Segment',
             color='white', fontsize=13, fontweight='bold')

# Panel 1: Break-Even
ax = axes[0]
for i, (label, stk, col) in enumerate(SCENARIOS_BAR):
    be_vals = []
    for ck, a in zip(capex_kwh_vals, arb_vals):
        net = a + stk - ck * 0.015
        be_vals.append(min(ck / net, 35) if net > 0 else 35)
    offset = (i - n_scen/2 + 0.5) * width
    ax.bar(x + offset, be_vals, width, label=label, color=col, alpha=0.87)
ax.axhline(LIFETIME, color='white', lw=1.5, ls='--', alpha=0.7,
           label=f'Ziel {LIFETIME}J')
ax.set_title('Break-Even [Jahre]', color='white')
ax.set_xticks(x); ax.set_xticklabels(seg_short, color='#aaa')
ax.set_ylabel('Jahre', color='#aaa')
ax.legend(fontsize=7.5, framealpha=0.35, facecolor='#111', labelcolor='white')
ax.grid(True, alpha=0.10, axis='y')

# Panel 2: ROI %
ax = axes[1]
ZIEL_ROI = round(100 / LIFETIME, 2)
for i, (label, stk, col) in enumerate(SCENARIOS_BAR):
    roi_vals = []
    for ck, a in zip(capex_kwh_vals, arb_vals):
        net = a + stk - ck * 0.015
        roi_vals.append(net / ck * 100 if ck > 0 else 0)
    offset = (i - n_scen/2 + 0.5) * width
    ax.bar(x + offset, roi_vals, width, label=label, color=col, alpha=0.87)
ax.axhline(ZIEL_ROI, color='white', lw=1.5, ls='--', alpha=0.7,
           label=f'Ziel-ROI {ZIEL_ROI}%/J')
ax.set_title('ROI [%/Jahr]', color='white')
ax.set_xticks(x); ax.set_xticklabels(seg_short, color='#aaa')
ax.set_ylabel('ROI %/Jahr', color='#aaa')
ax.legend(fontsize=7.5, framealpha=0.35, facecolor='#111', labelcolor='white')
ax.grid(True, alpha=0.10, axis='y')

plt.tight_layout()
out_path = os.path.join(CHARTS_DIR, 'kuer_nb09_fcr_gamechanger.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f'Gespeichert: {out_path}')

---
## 3. Visualisierung

### Chart 09: Erlösstacking — Vor/Nachher-Vergleich


In [ ]:
# ── Chart 09: Erlösstacking Balkendiagramm ────────────────────────────────────
# TODO: Vollständig implementieren
# Idee: Gestapelte Balken je Segment — Basis Arbitrage + je Erlösquelle
# Referenz Implementierung: ähnlich Chart 1d in NB03
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG_DARK)
for ax in axes:
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors='#bbbbbb')
    for sp in ax.spines.values(): sp.set_edgecolor('#333333')
fig.suptitle('Revenue Stacking: Erlöspotenzial mit VPP/FCR/aFRR',
             color='white', fontsize=13, fontweight='bold')

segs = list(df_econ['segment'])
x    = range(len(segs))
arb  = list(df_econ['rev_per_kwh'])
stk  = [total['pesimistisch'], total['mitte'], total['optimistisch']]

# Panel 1: Gestapelter Jahreserlös je Segment (mittleres Szenario)
ax = axes[0]
ax.bar(x, arb, label='Arbitrage', color='#FFA726', alpha=0.85)
ax.bar(x, [total['mitte']]*len(x), bottom=arb, label='FCR+aFRR+VPP', color='#66BB6A', alpha=0.85)
ax.axhline(20, color='white', lw=1, linestyle='--', alpha=0.5, label='Break-Even-Linie (~20 EUR/kWh/J)')
ax.set_title('Erlös/kWh/Jahr mit Stacking', color='white')
ax.set_xticks(list(x)); ax.set_xticklabels([s.split('_')[0] for s in segs], color='#aaa')
ax.set_ylabel('EUR/kWh/Jahr', color='#aaa')
ax.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
ax.grid(True, alpha=0.10, axis='y')

# Panel 2: Break-Even Vergleich
ax = axes[1]
capex_kwh_vals = [row['capex']/{'Privat_10kWh':10,'Gewerbe_100kWh':100,
                                  'Industrie_1MWh':1000,'Utility_10MWh':10000}[row['segment']]
                  for _, row in df_econ.iterrows()]
be_old = [ck / max(a - ck*0.015, 0.001) for ck, a in zip(capex_kwh_vals, arb)]
be_new = [ck / max(a + total['mitte'] - ck*0.015, 0.001) for ck, a in zip(capex_kwh_vals, arb)]
ax.bar([i - 0.2 for i in x], [min(b,30) for b in be_old], 0.35, label='Nur Arbitrage', color='#EF5350', alpha=0.85)
ax.bar([i + 0.2 for i in x], [min(b,30) for b in be_new], 0.35, label='Mit Stacking', color='#66BB6A', alpha=0.85)
ax.axhline(LIFETIME, color='#FFA726', lw=1.5, linestyle='--', alpha=0.7,
               label=f'Ziel-BE {LIFETIME}J')
ax.set_title('Break-Even Vergleich [Jahre]', color='white')
ax.set_xticks(list(x)); ax.set_xticklabels([s.split('_')[0] for s in segs], color='#aaa')
ax.set_ylabel('Break-Even [Jahre]', color='#aaa')
ax.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
ax.grid(True, alpha=0.10, axis='y')

plt.tight_layout()
out_path = os.path.join(CHARTS_DIR, 'kuer_nb09_revenue_stacking.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f'Gespeichert: {out_path}')

# ── Chart 09: Einzelplots ────────────────────────────────────────────────────
for fname, ax_src, title in [
    ('chart09a_stacking_erloese.png', axes[0], 'Erlös/kWh/Jahr mit Stacking'),
    ('chart09b_breakeven_vergleich.png', axes[1], 'Break-Even Vergleich [Jahre]'),
]:
    fig_e, ax_e = plt.subplots(figsize=(10, 6))
    fig_e.patch.set_facecolor('#0d1117'); ax_e.set_facecolor('#141414')
    ax_e.tick_params(colors='#bbbbbb')
    for sp in ax_e.spines.values(): sp.set_edgecolor('#333')
    for container in ax_src.containers:
        try:
            ax_e.bar([p.get_x()+p.get_width()/2 for p in container],
                     [p.get_height() for p in container],
                     width=container[0].get_width(),
                     color=[p.get_facecolor() for p in container], alpha=0.85)
        except Exception: pass
    for line in ax_src.get_lines():
        ax_e.axhline(line.get_ydata()[0], color=line.get_color(),
                     lw=line.get_linewidth(), linestyle=line.get_linestyle(),
                     label=line.get_label())
    ax_e.set_xticks(list(range(len(segs))))
    ax_e.set_xticklabels([s.split('_')[0] for s in segs], color='#aaa')
    ax_e.set_title(title, color='white')
    ax_e.set_ylabel(ax_src.get_ylabel() or '', color='#aaa')
    ax_e.grid(True, alpha=0.10, axis='y')
    if ax_src.get_legend(): ax_e.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_DIR, fname), dpi=DPI, bbox_inches='tight', facecolor='#0d1117')
    plt.close()
    print(f'  Einzelplot: {fname}')

---
## 4. Fazit

Mit VPP-Integration steigt der Netto-Jahreserlös einer 10-kWh-Batterie  
von ~50–100 EUR (reine Arbitrage) auf geschätzte **400–600 EUR/Jahr**  
→ Break-Even von >25 Jahre auf **7–10 Jahre** (mittleres Stacking-Szenario).

| Segment | Nur Arbitrage | Mit Stacking (Mitte) | Δ Break-Even |
|---------|--------------|----------------------|--------------|
| Privat | > 25 Jahre | ~10 Jahre | −15 Jahre |
| Gewerbe | ~12–18 Jahre | ~6–9 Jahre | −6 Jahre |
| Industrie | ~8–12 Jahre | ~4–6 Jahre | −4 Jahre |
| Utility | ~6–8 Jahre | ~3–5 Jahre | −3 Jahre |

> Vollständige Trigger-Synthese → **NB05 Business Strategy** Sektion 7 (VPP-Analyse).

---
| [← NB08 Marktdynamik](08_Marktdynamik.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB10 Dispatch-Optimierung](10_Dispatch_Optimierung.ipynb) |
|:---|:---:|---:|
